<a href="https://colab.research.google.com/github/RR77ui/Business-Intelligence/blob/main/Aprendizaje%20Supervisado/Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Parcial 2 - Redes neuronales

**Objetivo del taller**

Construir, entrenar y optimizar una red neuronal para clasificar la deserción de clientes (Churn: 1 = Sí, 0 = No) usando un dataset de una empresa de telecomunicaciones.

**Actividades a realizar**

1. Analizar el dataset

- ¿Qué variables pueden influir más en el churn?
- ¿El dataset está balanceado?

2. Construir y evaluar el modelo base
3. Probar mejoras:
- BatchNormalization
- Dropout
- Regularización L2
- EarlyStopping
- Aprendizaje por etapas (ajustar learning rate)

4. Comparar resultados
- ¿Qué técnica mejoró más?
- ¿Hubo sobreajuste? ¿Cómo lo detectaron?

5. Sacar conclusiones del negocio

Preguntas orientadoras:
- ¿Qué perfil de cliente tiene más probabilidad de irse?
- ¿Qué variables deberían monitorear en el área comercial?
- ¿Qué acciones recomendarían para retener clientes?

**Descripción de variables**

| **Columna**         | **Descripción**                                                                        |
|----------------------|----------------------------------------------------------------------------------------|
| Age                 | Edad del cliente                                                                       |
| Tenure              | Meses que lleva como cliente                                                           |
| MonthlyCharges      | Pago mensual actual                                                                    |
| TotalCharges        | Total pagado históricamente                                                            |
| Contract            | Tipo de contrato (Month-to-month, One year, Two year)                                  |
| InternetService     | Tipo de servicio de internet                                                            |
| TechSupport         | Si tiene soporte técnico (Yes / No)                                                    |
| PaymentMethod       | Método de pago                                                                         |
| Churn               | **Variable objetivo:** 1 = Cliente se fue / 0 = Cliente sigue                          |


0. Importamos las librerias

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Exploracion de datos

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Aprendizaje Supervisado/telecom_churn.csv')
print(data.shape)
print(data.info())
print(data['Churn'].value_counts())
data.head()


(1500, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              1500 non-null   int64  
 1   Tenure           1500 non-null   int64  
 2   MonthlyCharges   1500 non-null   float64
 3   TotalCharges     1500 non-null   float64
 4   Contract         1500 non-null   object 
 5   InternetService  1168 non-null   object 
 6   TechSupport      1500 non-null   object 
 7   PaymentMethod    1500 non-null   object 
 8   Churn            1500 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 105.6+ KB
None
Churn
1    897
0    603
Name: count, dtype: int64


,Age,Tenure,MonthlyCharges,TotalCharges,Contract,InternetService,TechSupport,PaymentMethod,Churn
0,63,8,69.42,526.45,Month-to-month,DSL,No,Bank transfer,1
1,20,9,49.63,565.06,Two year,Fiber optic,No,Electronic check,0
2,46,20,115.15,2288.21,One year,DSL,No,Mailed check,1
3,52,19,70.26,1298.68,Month-to-month,NaN,Yes,Electronic check,1
4,56,11,15.00,140.21,Two year,DSL,No,Electronic check,0


In [ ]:
# Fill missing InternetService values with a placeholder
data['InternetService'].fillna('No_internet_service', inplace=True)

# Identify categorical columns (excluding the target variable 'Churn')
categorical_cols = data.select_dtypes(include='object').columns.tolist()
# Assuming the user wants to encode all object type columns except 'Churn'
if 'Churn' in categorical_cols:
    categorical_cols.remove('Churn')

# Apply one-hot encoding to the categorical columns
datat = pd.get_dummies(data, columns=categorical_cols) # Removed drop_first=True

display(datat.head())

/tmp/ipython-input-3150150498.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['InternetService'].fillna('No_internet_service', inplace=True)


,Age,Tenure,MonthlyCharges,TotalCharges,Churn,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No_internet_service,TechSupport_No,TechSupport_Yes,PaymentMethod_Bank transfer,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,63,8,69.42,526.45,1,True,False,False,True,False,False,True,False,True,False,False,False
1,20,9,49.63,565.06,0,False,False,True,False,True,False,True,False,False,False,True,False
2,46,20,115.15,2288.21,1,False,True,False,True,False,False,True,False,False,False,False,True
3,52,19,70.26,1298.68,1,True,False,False,False,False,True,False,True,False,False,True,False
4,56,11,15.00,140.21,0,False,False,True,True,False,False,True,False,False,False,True,False


In [ ]:
correlation_matrix = datat.corr(numeric_only=True)
display(correlation_matrix)

,Age,Tenure,MonthlyCharges,TotalCharges,Churn,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No_internet_service,TechSupport_No,TechSupport_Yes,PaymentMethod_Bank transfer,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check
Age,1.000000,-0.055611,0.025438,-0.014905,0.047867,0.014110,-0.023526,0.009800,0.023446,-0.009570,-0.016566,-0.005861,0.005861,-0.041442,0.001596,0.037522,0.004781
Tenure,-0.055611,1.000000,-0.007509,0.765671,-0.046228,-0.018779,0.024718,-0.004768,0.009759,-0.014774,0.005682,-0.056288,0.056288,0.037314,-0.006541,-0.030757,-0.002076
MonthlyCharges,0.025438,-0.007509,1.000000,0.549163,0.247122,-0.023108,0.033839,-0.010162,-0.013404,-0.024337,0.044213,0.011053,-0.011053,-0.007122,-0.011154,0.006276,0.012454
TotalCharges,-0.014905,0.765671,0.549163,1.000000,0.110285,-0.019385,0.033598,-0.015065,-0.005041,-0.015821,0.024400,-0.024625,0.024625,0.009716,-0.010302,-0.014345,0.014143
Churn,0.047867,-0.046228,0.247122,0.110285,1.000000,0.676683,-0.486600,-0.335384,0.000039,-0.004153,0.004795,0.349540,-0.349540,0.036828,-0.008200,-0.026612,-0.003877
Contract_Month-to-month,0.014110,-0.018779,-0.023108,-0.019385,0.676683,1.000000,-0.720916,-0.493345,-0.011405,0.008432,0.003657,0.006687,-0.006687,0.022109,0.020002,-0.040672,-0.003740
Contract_One year,-0.023526,0.024718,0.033839,0.033598,-0.486600,-0.720916,1.000000,-0.247154,0.024435,-0.012021,-0.014879,0.012255,-0.012255,0.007054,0.003847,-0.019460,0.007545
Contract_Two year,0.009800,-0.004768,-0.010162,-0.015065,-0.335384,-0.493345,-0.247154,1.000000,-0.014722,0.003298,0.013562,-0.024731,0.024731,-0.039765,-0.032795,0.081291,-0.004241
InternetService_DSL,0.023446,0.009759,-0.013404,-0.005041,0.000039,-0.011405,0.024435,-0.014722,1.000000,-0.637261,-0.439558,0.010274,-0.010274,0.061037,-0.032342,-0.041683,0.010077
InternetService_Fiber optic,-0.009570,-0.014774,-0.024337,-0.015821,-0.004153,0.008432,-0.012021,0.003298,-0.637261,1.000000,-0.412094,0.006263,-0.006263,-0.030219,0.029626,0.011464,-0.009884


2. Modelo Simple

In [ ]:
X = datat.drop('Churn', axis=1)
y = datat['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("Calculated class weights:", class_weight_dict)

Calculated class weights: {0: np.float64(1.2041284403669725), 1: np.float64(0.8550488599348535)}


In [ ]:
# transformacion de la variables numericas para el modelo
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Modelo base simple
model = keras.Sequential([ #Crea un modelo secuencial en donde una capa esta apilada despues de la otra
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #Se crea una capa de 64 neuronas, con relu como función de activación
    layers.Dense(1, activation='sigmoid') # sigmoide para clasificacion
])
# Compilar modelo
model.compile(optimizer='adam',
              loss='binary_crossentropy', # Use binary_crossentropy for binary classification
              metrics=['accuracy'])
# Entrenar
history = model.fit(X_train, y_train, epochs=10, validation_split=0.3, class_weight=class_weight_dict)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.4919 - loss: 0.7074 - val_accuracy: 0.7333 - val_loss: 0.5931
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8200 - loss: 0.5315 - val_accuracy: 0.8286 - val_loss: 0.4890
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8695 - loss: 0.4395 - val_accuracy: 0.8476 - val_loss: 0.4175
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8981 - loss: 0.3664 - val_accuracy: 0.8381 - val_loss: 0.3728
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9017 - loss: 0.3201 - val_accuracy: 0.8444 - val_loss: 0.3381
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8867 - loss: 0.2819 - val_accuracy: 0.8508 - val_loss: 0.3122
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9105 - loss: 0.2392 - val_accuracy: 0.8603 - val_loss: 0.2966
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9022 - loss: 0.2328 - val_accuracy: 0.8571 - val_loss

2.1 Modelo de regresion logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Modelo de regresion logistica
model_regl = LogisticRegression(max_iter=1000)
model_regl.fit(X_train, y_train)

# Prediccion en el set de prueba
y_pred = model_regl.predict(X_test)

# Evaluacion del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(conf_matrix)

Accuracy: 0.9133333333333333

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.92      0.89       167
           1       0.95      0.91      0.93       283

    accuracy                           0.91       450
   macro avg       0.90      0.91      0.91       450
weighted avg       0.92      0.91      0.91       450


Confusion Matrix:
[[153  14]
 [ 25 258]]


In [ ]:
# Get the intercept and coefficients from the trained model
intercept = model_regl.intercept_[0]
coefficients = model_regl.coef_[0]

print(f"Intercept (β0): {intercept:.4f}")
print("\nCoefficients (βi):")
# Assuming X_train_scaled is a numpy array, get feature names from original X_train
feature_names = X.columns # Get column names from the original DataFrame X

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.4f}")

print("\nThe logistic regression equation (in terms of log-odds) is:")
print(f"ln(P(Personal.Loan=1) / (1 - P(Personal.Loan=1))) = {intercept:.4f} + ", end="")

for i, (feature, coef) in enumerate(zip(feature_names, coefficients)):
    if i > 0:
        print(" + ", end="")
    print(f"({coef:.4f} * {feature})", end="")

print()

Intercept (β0): 1.4435

Coefficients (βi):
Age: 0.1375
Tenure: 0.3800
MonthlyCharges: 2.0348
TotalCharges: -0.4674
Contract_Month-to-month: 2.3941
Contract_One year: -1.6003
Contract_Two year: -1.3351
InternetService_DSL: -0.0232
InternetService_Fiber optic: 0.0345
InternetService_No_internet_service: -0.0131
TechSupport_No: 1.5382
TechSupport_Yes: -1.5382
PaymentMethod_Bank transfer: 0.1097
PaymentMethod_Credit card: -0.0524
PaymentMethod_Electronic check: -0.2127
PaymentMethod_Mailed check: 0.1421

The logistic regression equation (in terms of log-odds) is:
ln(P(Personal.Loan=1) / (1 - P(Personal.Loan=1))) = 1.4435 + (0.1375 * Age) + (0.3800 * Tenure) + (2.0348 * MonthlyCharges) + (-0.4674 * TotalCharges) + (2.3941 * Contract_Month-to-month) + (-1.6003 * Contract_One year) + (-1.3351 * Contract_Two year) + (-0.0232 * InternetService_DSL) + (0.0345 * InternetService_Fiber optic) + (-0.0131 * InternetService_No_internet_service) + (1.5382 * TechSupport_No) + (-1.5382 * TechSupport_Yes)

3. Modelo Keras bach

In [ ]:
model_bn = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(1, activation='sigmoid') # Changed to 1 unit for binary classification
])

model_bn.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

history_bn = model_bn.fit(X_train, y_train, epochs=10, validation_split=0.3,class_weight=class_weight_dict)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7648 - loss: 0.4636 - val_accuracy: 0.8381 - val_loss: 0.4731
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9147 - loss: 0.1881 - val_accuracy: 0.8540 - val_loss: 0.4001
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9169 - loss: 0.1895 - val_accuracy: 0.8857 - val_loss: 0.3613
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9208 - loss: 0.1979 - val_accuracy: 0.8794 - val_loss: 0.3393
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9169 - loss: 0.1765 - val_accuracy: 0.8762 - val_loss: 0.3177
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9410 - loss: 0.1419 - val_accuracy: 0.8730 - val_loss: 0.2978
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9289 - loss: 0.1583 - val_accuracy: 0.8794 - val_loss: 0.2850
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9435 - loss: 0.1350 - val_accuracy: 0.8794 - val_loss: 0.2

4. Modelo Dropout

In [ ]:
model_dropout = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),     # Apaga el 30% de neuronas
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='sigmoid')
])

model_dropout.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

history_dropout = model_dropout.fit(X_train, y_train, epochs=10, validation_split=0.3,class_weight=class_weight_dict)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3659 - loss: 1.8444 - val_accuracy: 0.8032 - val_loss: 0.8244
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7966 - loss: 0.7209 - val_accuracy: 0.8508 - val_loss: 0.4218
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8551 - loss: 0.4130 - val_accuracy: 0.8698 - val_loss: 0.3068
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8752 - loss: 0.3092 - val_accuracy: 0.8730 - val_loss: 0.2953
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8686 - loss: 0.2598 - val_accuracy: 0.8698 - val_loss: 0.3024
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8982 - loss: 0.2225 - val_accuracy: 0.8762 - val_loss: 0.2836
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8944 - loss: 0.2187 - val_accuracy: 0.8730 - val_loss: 0.2895
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9153 - loss: 0.1974 - val_accuracy: 0.8762 - val_loss

5. Modelo earlystopping

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=2, restore_best_weights=True
)

model_es = keras.Sequential([
    layers.Dense(128, activation='relu',input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='sigmoid')
])

model_es.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

history_es = model_es.fit(X_train, y_train,
                          epochs=10, validation_split=0.3,
                          callbacks=[early_stop],class_weight=class_weight_dict)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3093 - loss: 2.0773 - val_accuracy: 0.8159 - val_loss: 0.9200
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8303 - loss: 0.7294 - val_accuracy: 0.8349 - val_loss: 0.4190
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8784 - loss: 0.3507 - val_accuracy: 0.8508 - val_loss: 0.3046
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8788 - loss: 0.2510 - val_accuracy: 0.8603 - val_loss: 0.2703
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8705 - loss: 0.2669 - val_accuracy: 0.8825 - val_loss: 0.2649
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9083 - loss: 0.2028 - val_accuracy: 0.8825 - val_loss: 0.2521
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9103 - loss: 0.2068 - val_accuracy: 0.8921 - val_loss: 0.2546
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9104 - loss: 0.1831 - val_accuracy: 0.8794 - val_loss

6. Modelo Turned

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=0.0005)

model_tuned = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='sigmoid')
])

model_tuned.compile(optimizer=optimizer,
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

history_tuned = model_tuned.fit(X_train, y_train,
                                epochs=10, batch_size=64,
                                validation_split=0.3,class_weight=class_weight_dict)

Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.2428 - loss: 2.1999 - val_accuracy: 0.4762 - val_loss: 1.9176
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5061 - loss: 1.4641 - val_accuracy: 0.7175 - val_loss: 1.6831
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7278 - loss: 0.9517 - val_accuracy: 0.8000 - val_loss: 1.4707
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8320 - loss: 0.6265 - val_accuracy: 0.8159 - val_loss: 1.2909
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8565 - loss: 0.4890 - val_accuracy: 0.8317 - val_loss: 1.1378
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8740 - loss: 0.4022 - val_accuracy: 0.8349 - val_loss: 1.0119
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8902 - loss: 0.3364 - val_accuracy: 0.8381 - val_loss: 0.9043
Epoch 8/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8798 - loss: 0.3010 - val_accuracy: 0.8571 - val_loss

7. Resumen de metricas

In [ ]:
models = {
    "Base": model,
    "BatchNorm": model_bn,
    "Dropout": model_dropout,
    "EarlyStopping": model_es,
    "Tuned": model_tuned
}

print(f"{'Model':15s} → Precisión | Loss")
print("-" * 30)

for name, m in models.items():
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{name:15s} → Precision:{acc:.4f}    | Perdida:{loss:.4f}")

Model           → Precisión | Loss
------------------------------
Base            → Precision:0.8756    | Perdida:0.2615
BatchNorm       → Precision:0.8756    | Perdida:0.2583
Dropout         → Precision:0.8822    | Perdida:0.2476
EarlyStopping   → Precision:0.8933    | Perdida:0.2355
Tuned           → Precision:0.8689    | Perdida:0.6607


##**Analisis de Resultado**
1. Se hizo la exploracion de los datos y se encontro que la variable a predecir tiene un pequeño desbalanceo de clase debido a que tiene mas valores de 1 que de 0 alrededor de 200 de diferencia lo que podria afectar el modelo para predecir sin embargo se hicieron pruebas con y sin balanceo de clases y los resultados fueron muy parecidos por lo cual se dejo el balanceo para evitar problemas

2. Analizando los coeficientes de la regresion podemos ver que las variables que mayor peso tienen a la hora de predecir son los monthly charges, el tipo de contrato que el cliente siendo el de mayor peso month to month y por ultimo si posee o no soporte tecnico serian las variables mas importantes para saber si un cliente va tiene mayor probabilidad de irse o no

3. Se realizaron los siguientes modelos con los siguientes resultados de precision: Regresion logistica-91%, Modelo simple-87%, Modelo Bach-87%, Modelo Dropout-88%, Modelo Earlystopping-89%, Modelo Turned-86%. Como se puede evidenciar el de peor desempeño fue el modelo turned y el mejor fue la regresion logistica sin embargo se tienen en deberia tener en cuenta tambien el modelo de earlystopping para la prediccion junto con la regresion logistica que ayuda a dar mas insigths importantes para el negocio. se vio un poco de sobreajuste principalmente en el modelo bach que se evidencio con la diferencia entre el train y test que fue de mas del 5% lo que muestra un desempeño diferente entre el entrenamiento y test

4. Respondiendo a las preguntas de negocio, el perfil de cliente que mas probabilidad tiene de irse es los que tiene monthly charges muy altos, los que tienen contratos de month to month y lo que no cuentan con tech support ya que esta variable son las que tienen mayor coefieciente positivo dentro de la ecuacion ademas de una correlacion de 24%, 67% y 34% respectivamente, estas son las variables que mas deberia monitorear el area comercial para poder hacer ofertas mas personalizadas y que ayuden a retener mas clientes, tambien deberian incentivar a contratos mas largos de 1 a 2 años y promocinar de mejor manera el soporte tecnico.